In [1]:
%cd ..

/Users/n-zagainov/kolobok/ml


In [2]:
from pathlib import Path

import numpy as np
import cv2
import torch
from shapely.geometry import Polygon
from matplotlib import pyplot as plt


from tire_vision.text.preprocessor.model import TireDetector
from tire_vision.text.preprocessor.unwrapper import TireUnwrapper
from tire_vision.config import TireVisionConfig


[07/01/25 06:59:01] WARNING  Your inference package version 0.51.0 is out of date! Please upgrade to ]8;id=662229;file:///Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=538339;file:///Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/core/__init__.py#41\41]8;;\
                             version 0.51.1 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:341: ModelDependencyMissing: Your `inference` configuration does not support Qwen2.5-VL model. Use pip install 'inference[transformers]' to install missing requirements.
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:353: ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:363: ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/inference/models/utils.py:374: ModelDependencyMissing: Your `inference` configuration does no

In [3]:
cfg = TireVisionConfig()
detector = TireDetector(cfg.tire_detector)
unwrapper = TireUnwrapper(cfg.tire_unwrapper)

/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:69: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.9/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:69: UserWarning: Specified provider 'OpenVINOExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [4]:
def show_image(image, figsize=(10, 8), title=None):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    if title is not None:
        plt.title(title)
    plt.axis('off')
    plt.show()

In [5]:
root = Path("data/parking")

imgs = [cv2.imread(str(p)) for p in sorted(root.glob("*.jpg"))][2:]


In [6]:
output = []
for input_image in imgs:
    try:
        detection = detector.detect(input_image)
        unwrapped = unwrapper.get_unwrapped_tire(input_image, detection["wheel"], detection["rim"])

        output.append(unwrapped)
    except Exception as e:
        print(e)
        continue

In [7]:
output_dir = Path("data/parking_output")
output_dir.mkdir(parents=True, exist_ok=True)

for i, img in enumerate(output):
    cv2.imwrite(str(output_dir / f"{i}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

In [9]:
# from predictions import getPredictions

# img_results, entities = getPredictions(cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

In [10]:
IMAGE_PATH = "/Users/n-zagainov/kolobok/ml/data/parking_output/2.png"

In [13]:
import sys
sys.path.append(str(Path.cwd() / "external" / "surya"))

In [27]:
from PIL import Image
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

image = Image.open(IMAGE_PATH)
recognition_predictor = RecognitionPredictor()
detection_predictor = DetectionPredictor()

predictions = recognition_predictor([image], det_predictor=detection_predictor)

Recognizing Text: 100%|██████████| 15/15 [00:25<00:00,  1.68s/it]


In [31]:
[item.text for item in predictions[0].text_lines]

['The same of the',
 '<b>COLLECTIVE</b>',
 'SMERT HEX AVE S',
 '',
 '(B.P.R. BANGE H TREAD 4 PLIES STEEL',
 'REGROOVABLE ANDIAL',
 '<math>M+S</math><br>TRACTION',
 '<math>\\overline{</math>DOT (<math>\\overline{R}</math>J.5K) 3TH (3422) 15316 MADE',
 'AD SINGLE = 2000 kg (4.410 LBS) = AT 775 kPa (110 PSI) = COL<br>AD DUAL = = 1900 kg (4190 LBS) = AT 775 kPa (110 PSI) = COL',
 'SMERT FILX AVA SA',
 '<b>Community Community Community Community Community</b>',
 '<sup>4 PLIES</sup> STEEL',
 'REGROOVABLE Anti- M+S<br>TUBELESS RADIAL TRACTION',
 '235/75R17.5',
 'H(3422)']